In [ ]:
!git clone -b robs-branch --single-branch https://github.com/RobertDavidson1/GrokkingExperiments.git
%cd /content/GrokkingExperiments

%pip install --no-deps -e .
%pip install --no-deps scienceplots


import sys
repo = Path("/content/GrokkingExperiments")
package = next(repo.rglob("grokkingexperiments/config.py"))

sys.path.insert(0, str(package.parent.parent))

fatal: destination path 'GrokkingExperiments' already exists and is not an empty directory.
/content/GrokkingExperiments
Obtaining file:///content/GrokkingExperiments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grokkingexperiments (pyproject.toml) ... done
  Created wheel for grokkingexperiments: filename=grokkingexperiments-0.1.0-py3-none-any.whl size=3482 sha256=d28ee81ddde10c96945f4514b0b40cab567511e36922f488a188f0a2babff7e9
  Stored in directory: /tmp/pip-ephem-wheel-cache-89ljz3fj/wheels/d3/e5/64/e6e5467af90300195e6acf9aa85f648301f26229be08833fdd
Successfully built grokkingexperiments
  Attempting uninstall: grokkingexperiments
    Found existing installation: grokkingexperiments 0.1.0
    Uninstalling grokkingexperiments-0.1.0:
      Successfully uninstalled grokkingexperiments-0.1.0


In [5]:
from grokkingexperiments.config import print_config, load_config
from grokkingexperiments.model import GrokkingModel
from grokkingexperiments.groups.zn import Zn

import torch
import numpy as np
import matplotlib.pyplot as plt

import scienceplots
plt.style.use(["science", "ieee", "notebook"])
# plt.style.use(["science", "notebook"])


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} to train")

ModuleNotFoundError: No module named 'grokkingexperiments'

In [ ]:
config = load_config()
config.train.epochs = 250_000
config.settings.progress_interval = 10_000
config.group.n = 120

print_config(config)




In [ ]:
left_embed_weights = {}
right_embed_weights = {}

n_seeds = 150

for seed in range(n_seeds):
    print(f"Training model with seed {seed}")
    config.settings.seed = seed # Overwrite seed

    model = GrokkingModel(config).to(device)
    zn = Zn(config)

    # Generate the data
    x_train, y_train, x_test, y_test = zn.generate_train_test_data()

    # Move to GPU if available
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    x_test = x_test.to(device)
    y_test = y_test.to(device)

    # Train the model
    model.fit(x_train, y_train, x_test, y_test)


    # Store both embeddings on CPU
    E_L = model.state_dict()["E_L"]
    left_embed_weights[seed] = E_L.detach().cpu().clone()

    E_R = model.state_dict()["E_R"]
    right_embed_weights[seed] = E_R.detach().cpu().clone()


embed_weights = {
    "left": left_embed_weights,
    "right": right_embed_weights
}

torch.save(embed_weights, "embed_weights.pt")

In [ ]:
from google.colab import files
files.download("embed_weights.pt")


embed_weights = torch.load(
    "embed_weights.pt",
    map_location="cpu",
    weights_only=True
)